In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
project_root = os.getenv('MED_AI_PROJECT_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))
if not os.path.exists(os.path.join(project_root, 'dataset')):
    project_root = os.getcwd()
dataset_dir = os.path.join(project_root, 'dataset')
metadata_csv = os.path.join(dataset_dir, 'HAM10000_metadata.csv')
images_dir = os.path.join(dataset_dir, 'images')
feedback_dir = os.getenv('MED_AI_FEEDBACK_DIR', os.path.join(project_root, 'feedback_data1'))
feedback_csv_path = os.path.join(feedback_dir, 'clinical_feedback_log.csv')
model_epoch3_path = os.getenv('MED_AI_MODEL_EPOCH3_PATH', os.path.join(project_root, 'outputs', 'checkpoints', 'eca_resnet50_epoch3.pth'))
model_best_path = os.getenv('MED_AI_MODEL_BEST_PATH', os.path.join(project_root, 'outputs', 'checkpoints', 'eca_resnet50_best.pth'))
model_v2_best_path = os.getenv('MED_AI_MODEL_V2_BEST_PATH', os.path.join(project_root, 'outputs', 'checkpoints', 'eca_resnet50_v2_best.pth'))
model_v2_finetuned_path = os.getenv('MED_AI_MODEL_V2_FINETUNED_PATH', os.path.join(project_root, 'outputs', 'checkpoints', 'eca_resnet50_v2_finetuned.pth'))
model_v8_best_path = os.getenv('MED_AI_MODEL_V8_BEST_PATH', os.path.join(project_root, 'outputs', 'checkpoints', 'eca_resnet50_v8_best_acc87_97.pth'))
global_seed = int(os.getenv('MED_AI_SEED', '42'))

random.seed(global_seed)
np.random.seed(global_seed)
torch.manual_seed(global_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(global_seed)


transform = transforms.Compose([
    transforms.ToTensor()
]) # 转换Tensor 格式

# 1. 定义数据预处理流水线 
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),      # 强制缩放到 ResNet50 的标准输入尺寸
    transforms.RandomHorizontalFlip(),  # 随机水平翻转
    transforms.RandomVerticalFlip(),    # 随机垂直翻转
    transforms.PILToTensor(),          # 纯 PIL 转 Tensor，避免依赖 torch.from_numpy
    transforms.ConvertImageDtype(torch.float32),  # 转为 float32 并缩放到 [0,1]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # 官方要求的标准化参数
])

# 2. 编写自定义医疗数据集类
class ISICDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        """
        csv_file: 标签表格的路径
        img_dir: 图片文件夹的路径
        transform: 数据预处理操作
        """
        self.labels_df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        # 告诉模型总共有多少张图片
        return len(self.labels_df)

    def __getitem__(self, idx):
        # 根据索引找到图片的文件名
        img_name = self.labels_df.iloc[idx, 0] + '.jpg'
        img_path = os.path.join(self.img_dir, img_name)

        # 读取图片并强制转换为 RGB 色彩模式
        image = Image.open(img_path).convert('RGB')

        # 获取对应的疾病标签（ISIC表格后面几列是疾病分类的 one-hot 编码，我们把它转成单个数字标签）
        labels = self.labels_df.iloc[idx, 1:].values.astype('float32')
        label = torch.argmax(torch.tensor(labels))

        # 执行前面定义的裁剪和翻转操作
        if self.transform:
            image = self.transform(image)

        return image, label

print("✅ 数据集类 ISICDataset 定义成功！")

In [ ]:
import pandas as pd
import numpy as np
import torch

# 1. 重新读取表格数据
csv_path = metadata_csv
df = pd.read_csv(csv_path)

# 重新建立标签映射
disease_classes = {
    'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3, 
    'akiec': 4, 'vasc': 5, 'df': 6
}
df['label_idx'] = df['dx'].map(disease_classes)

# 2. 统计每种疾病的样本数量
class_counts = df['label_idx'].value_counts().sort_index().values
total_samples = len(df)
num_classes = len(class_counts)

print(f"📊 各类别样本数量分布: {class_counts}")

# 3. 计算类别权重公式
weights = total_samples / (num_classes * class_counts)

# 4. 把权重转换成显卡能认识的 Tensor 格式
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.FloatTensor(weights).to(device)

print("\n🎯 自动计算出的类别惩罚权重:")
for i, w in enumerate(weights):
    print(f"类别 {i} (对应 {list(disease_classes.keys())[i]}) 权重: {w:.4f}")

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# 1. 配置文件路径
csv_path = metadata_csv
img_dir = images_dir

# 2. 读取表格并建立标签映射
df = pd.read_csv(csv_path)
disease_classes = {
    'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3, 
    'akiec': 4, 'vasc': 5, 'df': 6
}
df['label_idx'] = df['dx'].map(disease_classes)

# 3. 编写 HAM10000 专属数据集类
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 提取图片名字和标签
        img_name = self.df.iloc[idx]['image_id'] + '.jpg'
        img_path = os.path.join(self.img_dir, img_name)
        label = self.df.iloc[idx]['label_idx']

        # 读取图片
        image = Image.open(img_path).convert('RGB')

        # 🌟 修复 1：正确的预处理调用逻辑
        if self.transform:
            image = self.transform(image)

        return image, label

# 4. 设置预处理流水线
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.PILToTensor(),      
    transforms.ConvertImageDtype(torch.float32) 
])

# 实例化数据集和 DataLoader
dataset = HAM10000Dataset(df, img_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# 5. 测试能否成功取出一批数据
images, labels = next(iter(dataloader))

print(f"🎉 成功取出 1 批数据！")
print(f"图片张量形状: {images.shape} (代表: 数量, 通道数, 高度, 宽度)")

# 用 matplotlib 画出这 4 张图
plt.figure(figsize=(12, 3))
for i in range(4):
    plt.subplot(1, 4, i+1)
    img_plot = transforms.ToPILImage()(images[i].cpu())
    plt.imshow(img_plot)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')
plt.show()

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import math

# ==========================================
# 定义 ECA 轻量化通道注意力模块
# ==========================================
class eca_layer(nn.Module):
    """Constructs a ECA module."""
    def __init__(self, channel, b=1, gamma=2):
        super(eca_layer, self).__init__()
        # 自适应计算一维卷积核大小，替代庞大的全连接层
        t = int(abs((math.log(channel, 2) + b) / gamma))
        k = t if t % 2 else t + 1
        
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: input features with shape [b, c, h, w]
        b, c, h, w = x.size()
        y = self.avg_pool(x)
        y = self.conv(y.squeeze(-1).transpose(-1, -2)).transpose(-1, -2).unsqueeze(-1)
        y = self.sigmoid(y)
        return x * y.expand_as(x)

# ==========================================
# 将 ECA 模块融合进 ResNet50 主干网络
# ==========================================
class ECAResNet50(nn.Module):
    def __init__(self, num_classes=7):
        super(ECAResNet50, self).__init__()
        # 加载官方预训练的 ResNet50 权重
        weights = models.ResNet50_Weights.DEFAULT
        self.resnet = models.resnet50(weights=weights)

        # 剥离原模型的主干层
        self.conv1 = self.resnet.conv1
        self.bn1 = self.resnet.bn1
        self.relu = self.resnet.relu
        self.maxpool = self.resnet.maxpool
        
        self.layer1 = self.resnet.layer1
        self.eca1 = eca_layer(256)  # 插入注意力层1
        
        self.layer2 = self.resnet.layer2
        self.eca2 = eca_layer(512)  # 插入注意力层2
        
        self.layer3 = self.resnet.layer3
        self.eca3 = eca_layer(1024) # 插入注意力层3
        
        self.layer4 = self.resnet.layer4
        self.eca4 = eca_layer(2048) # 插入注意力层4
        
        self.avgpool = self.resnet.avgpool
        # 修改最后的全连接分类层：输出7种皮肤病变
        self.fc = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.eca1(x) # 注意力机制生效

        x = self.layer2(x)
        x = self.eca2(x) 

        x = self.layer3(x)
        x = self.eca3(x) 

        x = self.layer4(x)
        x = self.eca4(x) 

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# ==========================================
# 实例化并测试 算力
# ==========================================
# 1. 召唤模型
model = ECAResNet50(num_classes=7)

# 2. 自动识别你的显卡并把模型装载上去
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✅ 基于 ECA-Net 注意力机制的 ResNet50 构建成功！")
print(f"🔥 当前使用的计算设备: {device} ")

# 3. 模拟一张图片送进去测试一下能不能跑通
dummy_input = torch.randn(1, 3, 224, 224).to(device) # 模拟 1 张 224x224 的彩色图片
output = model(dummy_input)
print(f"📊 模型输出的张量形状: {output.shape} (代表: 1张图片, 7个类别的预测概率得分)")

In [ ]:
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

def evaluate_multiclass_metrics(logits_tensor, labels_tensor, target_name_list):
    probabilities = np.array(torch.softmax(logits_tensor, dim=1).cpu().tolist(), dtype=np.float32)

    y_true = np.array(labels_tensor.cpu().tolist(), dtype=np.int64)
    y_pred = probabilities.argmax(axis=1)
    # 精确率：预测为有病的人中，真正有病的比例
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    # 召回率：所有真正有病的人中，被你模型找出来的比例。
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    # 精确率和召回率的调和平均数，综合考虑了两者的表现
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    # AUC(ROC 曲线下面积)：衡量模型区分不同类别的能力。
    try:
        auc_macro_ovr = roc_auc_score(y_true, probabilities, multi_class='ovr', average='macro')
    except ValueError:
        auc_macro_ovr = float('nan')
    #生成可视化报告和矩阵    
    report_text = classification_report(y_true, y_pred, target_names=target_name_list, digits=4, zero_division=0)
    matrix = confusion_matrix(y_true, y_pred)
    return precision_macro, recall_macro, f1_macro, auc_macro_ovr, report_text, matrix





# ==========================================
# 1. 划分数据集：80% 训练，20% 测试
# ==========================================
# 使用分层抽样 (stratify)，保证每种疾病在训练和测试中的比例一致
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_idx'])

# 重新打包数据集
train_dataset = HAM10000Dataset(train_df, img_dir, transform=transform)
val_dataset = HAM10000Dataset(val_df, img_dir, transform=transform)

# 加载
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# ==========================================
# 2. 根据损失函数优化
# ==========================================
# 交叉熵损失函数
criterion = torch.nn.CrossEntropyLoss() 
# 优化器 Adam
optimizer = optim.Adam(model.parameters(), lr=0.0001) 

# ==========================================
# 3. 正式开始训练大循环
# ==========================================
num_epochs = 3 # 测试 3 轮 (Epoch)

print("🚀 显卡引擎轰鸣中... 开始训练模型！")
print("👀 请观察 Loss（误差）是否逐渐变小，Acc（准确率）是否逐渐变大...")

for epoch in range(num_epochs):
    # --- A. 开启训练模式 ---
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0
    
    # 遍历训练集里的每一批图片
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device) # 把图片和标准答案送到显卡
        
        optimizer.zero_grad() # 清空上一次的记忆
        
        outputs = model(inputs) # 1. 前向传播：让模型看图并给出预测
        loss = criterion(outputs, labels) # 2. 计算误差：对比模型的预测和标准答案
        
        loss.backward() # 3. 反向传播：分析错题原因（求导）
        optimizer.step() # 4. 更新参数
        
        # 统计数据用于展示
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # 每看 50 批图片（大概1600张），汇报一次平时成绩
        if (i+1) % 50 == 0:
            print(f"  [第 {epoch+1} 轮, 进度 {i+1} 批] 平均误差(Loss): {running_loss/50:.4f}, 平时练习准确率(Acc): {100 * correct / total:.2f}%")
            running_loss = 0.0
            correct = 0
            total = 0
            
    # --- B. 开启测试（验证集） ---
    model.eval() # 不准修改参数
    val_correct = 0
    val_total = 0
    val_logits_buffer = []
    val_labels_buffer = []
    with torch.no_grad(): # 测试不计算梯度，省下大量显卡资源
        for val_inputs, val_labels in val_loader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
            val_outputs = model(val_inputs)
            _, val_predicted = torch.max(val_outputs.data, 1)
            val_total += val_labels.size(0)
            val_correct += (val_predicted == val_labels).sum().item()
            val_logits_buffer.append(val_outputs.detach().cpu())
            val_labels_buffer.append(val_labels.detach().cpu())
            
    val_logits_tensor = torch.cat(val_logits_buffer, dim=0)
    val_labels_tensor = torch.cat(val_labels_buffer, dim=0)
    precision_macro, recall_macro, f1_macro, auc_macro_ovr, val_report, val_matrix = evaluate_multiclass_metrics(val_logits_tensor, val_labels_tensor, list(disease_classes.keys()))
    print(f"🏆 第 {epoch+1} 轮完整结束 | 📝 验证准确率: {100 * val_correct / val_total:.2f}% | Precision(macro): {precision_macro:.4f} | Recall(macro): {recall_macro:.4f} | F1(macro): {f1_macro:.4f}")
    if not np.isnan(auc_macro_ovr):
        print(f"📈 第 {epoch+1} 轮 AUC(OVR, macro): {auc_macro_ovr:.4f}")
    if epoch == num_epochs - 1:
        print("\n📌 最终验证集分类报告：")
        print(val_report)
        print("📌 最终验证集混淆矩阵：")
        print(val_matrix)
    print()

print("🎉 前 3 轮试运行训练完成！你可以保存这个模型了！")

In [ ]:
# 设定保存路径（保存在你熟悉的工作目录）
save_path = model_epoch3_path
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# 提取模型的大脑权重并保存到硬盘
torch.save(model.state_dict(), save_path)

print(f"💾 存档成功！模型权重已保存至: {save_path}")

In [8]:
# ==========================================
# 进阶训练：冲刺高分与最优模型保存
# ==========================================
num_epochs = 15 
best_model_path = model_v2_best_path  # fallback path if no new record in this run
last_saved_model_path = None

import re
from pathlib import Path

def get_next_versioned_model_path(seed_path, best_acc_value):
    model_dir = Path(seed_path).parent
    pattern = re.compile(r"eca_resnet50_v(\d+)_best(?:_acc\d+_\d+)?\.pth$")
    max_version = 1
    for p in model_dir.glob("eca_resnet50_v*_best*.pth"):
        m = pattern.match(p.name)
        if m:
            max_version = max(max_version, int(m.group(1)))
    next_version = max_version + 1
    acc_tag = f"{best_acc_value:.2f}".replace(".", "_")
    next_path = model_dir / f"eca_resnet50_v{next_version}_best_acc{acc_tag}.pth"
    return next_path, next_version, acc_tag



# 逻辑：只要你的 Jupyter 不重启，它就会一直把最高分记在内存里。
if 'best_val_acc' not in globals():
    best_val_acc = 0.0
    print("🆕 检测到全新训练，初始最高分门槛设为: 0.0%")
else:
    print(f"🔄 检测到内存中的历史记录！当前最高分门槛保持为: {best_val_acc:.2f}%")

# 🌟 带权重的交叉熵损失函数，少数类在计算损失时被放大
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

# 优化器
import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print(f"🚀 终极冲刺开始！总计 {num_epochs} 轮。")
print(f"[Policy] Train {num_epochs} epochs, pick run-best once, and save only if run-best > historical best ({best_val_acc:.2f}%).")
last_val_report = ""
last_val_matrix = None
run_best_val_acc = -1.0
run_best_epoch = -1
run_best_state_dict = None


for epoch in range(num_epochs):
    # --- A. 开启训练模式 ---
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels) 
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        if (i+1) % 50 == 0:
            print(f"  [第 {epoch+1} 轮, 进度 {i+1} 批] Loss: {running_loss/50:.4f}, 平时准确率: {100 * correct / total:.2f}%")
            running_loss = 0.0
            
    # --- B. 开启考试模式 ---
    model.eval()
    val_correct = 0
    val_total = 0
    val_logits_buffer = []
    val_labels_buffer = []
    with torch.no_grad():
        for val_inputs, val_labels in val_loader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
            val_outputs = model(val_inputs)
            _, val_predicted = torch.max(val_outputs.data, 1)
            val_total += val_labels.size(0)
            val_correct += (val_predicted == val_labels).sum().item()
            val_logits_buffer.append(val_outputs.detach().cpu())
            val_labels_buffer.append(val_labels.detach().cpu())
            
    val_logits_tensor = torch.cat(val_logits_buffer, dim=0)
    val_labels_tensor = torch.cat(val_labels_buffer, dim=0)
    precision_macro, recall_macro, f1_macro, auc_macro_ovr, val_report, val_matrix = evaluate_multiclass_metrics(val_logits_tensor, val_labels_tensor, list(disease_classes.keys()))
    last_val_report = val_report
    last_val_matrix = val_matrix
    current_val_acc = 100 * val_correct / val_total
    
    print(f"🏆 第 {epoch+1} 轮结束 | 📝 验证准确率: {current_val_acc:.2f}% | Precision(macro): {precision_macro:.4f} | Recall(macro): {recall_macro:.4f} | F1(macro): {f1_macro:.4f}")
    if not np.isnan(auc_macro_ovr):
        print(f"📈 第 {epoch+1} 轮 AUC(OVR, macro): {auc_macro_ovr:.4f}")
    
    # --- C. 智能存档最优模型 ---
    if current_val_acc > run_best_val_acc:
        print(f"   [Run-best updated] {run_best_val_acc:.2f}% -> {current_val_acc:.2f}% (record in memory, save once after full run).")
        run_best_val_acc = current_val_acc
        run_best_epoch = epoch + 1
        run_best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print("-" * 50)

print(f"[Run summary] Best val acc in this {num_epochs}-epoch run: {run_best_val_acc:.2f}% (epoch {run_best_epoch})")
if run_best_state_dict is None:
    print("[Warning] No model state captured in this run; no new versioned file created.")
elif run_best_val_acc > best_val_acc:
    best_model_path, best_model_version, best_model_acc_tag = get_next_versioned_model_path(model_v2_best_path, run_best_val_acc)
    best_model_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(run_best_state_dict, best_model_path)
    last_saved_model_path = best_model_path
    best_val_acc = run_best_val_acc
    print(f"[Saved] New historical best model: v{best_model_version}, acc={run_best_val_acc:.2f}% -> {best_model_path}")
else:
    print(f"[Not saved] Run-best {run_best_val_acc:.2f}% did not exceed historical best {best_val_acc:.2f}%.")
print("\n📌 最后一轮验证集分类报告：")
print(last_val_report)
print("📌 最后一轮验证集混淆矩阵：")
print(last_val_matrix)

  [第 12 轮, 进度 100 批] Loss: 0.0191, 平时准确率: 98.16%
  [第 12 轮, 进度 150 批] Loss: 0.0254, 平时准确率: 98.21%
  [第 12 轮, 进度 200 批] Loss: 0.0428, 平时准确率: 98.27%
  [第 12 轮, 进度 250 批] Loss: 0.0225, 平时准确率: 98.40%
🏆 第 12 轮结束 | 📝 验证准确率: 83.67% | Precision(macro): 0.6932 | Recall(macro): 0.6982 | F1(macro): 0.6744
📈 第 12 轮 AUC(OVR, macro): 0.9530
--------------------------------------------------
  [第 13 轮, 进度 50 批] Loss: 0.0441, 平时准确率: 98.88%
  [第 13 轮, 进度 100 批] Loss: 0.0245, 平时准确率: 99.00%
  [第 13 轮, 进度 150 批] Loss: 0.0510, 平时准确率: 98.77%
  [第 13 轮, 进度 200 批] Loss: 0.0797, 平时准确率: 98.06%
  [第 13 轮, 进度 250 批] Loss: 0.2004, 平时准确率: 96.94%
🏆 第 13 轮结束 | 📝 验证准确率: 80.48% | Precision(macro): 0.6365 | Recall(macro): 0.7854 | F1(macro): 0.6871
📈 第 13 轮 AUC(OVR, macro): 0.9658
--------------------------------------------------
  [第 14 轮, 进度 50 批] Loss: 0.0816, 平时准确率: 96.38%
  [第 14 轮, 进度 100 批] Loss: 0.0613, 平时准确率: 95.97%
  [第 14 轮, 进度 150 批] Loss: 0.0344, 平时准确率: 96.62%
  [第 14 轮, 进度 200 批] Loss: 0.0420, 平时准确率: 96.8

In [9]:
# 🌟 医生反馈闭环系统再训练
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import os
import subprocess
import sys

print("🔄 正在启动【医疗 AI 错题本增量学习 (闭环演进)】系统...")

# 0. 先生成扩展反馈训练池：真实反馈错题增强 + HAM10000 训练集难例挖掘
expand_script_path = os.path.join(project_root, 'scripts', 'expand_feedback_training_pool.py')
if os.path.exists(expand_script_path):
    try:
        subprocess.run([
            sys.executable,
            expand_script_path,
            '--project-root', project_root,
            '--feedback-dir', feedback_dir,
            '--model-path', model_v8_best_path,
            '--augment-per-image', '4',
            '--mine-max-count', '300',
            '--low-confidence-threshold', '0.80',
            '--device', 'auto'
        ], check=True)
    except Exception as exc:
        print(f"⚠️ 扩展反馈训练池生成失败，将回退到原始反馈候选集：{exc}")
else:
    print(f"⚠️ 未找到扩展反馈训练池脚本：{expand_script_path}")

# 1. 优先读取审核通过后的微调候选集；不存在时回退到反馈总表
feedback_csv_path = os.path.join(feedback_dir, 'clinical_feedback_log.csv')
finetune_expanded_csv_path = os.path.join(feedback_dir, 'finetune_candidates_expanded.csv')
finetune_candidates_csv_path = os.path.join(feedback_dir, 'finetune_candidates.csv')
if os.path.exists(finetune_expanded_csv_path):
    source_csv_path = finetune_expanded_csv_path
elif os.path.exists(finetune_candidates_csv_path):
    source_csv_path = finetune_candidates_csv_path
else:
    source_csv_path = feedback_csv_path

# 检查错题本是否存在，以及是否有误判记录
if not os.path.exists(source_csv_path):
    print("⚠️ 错题本目前为空，AI 暂时不需要加练！")
else:
    feedback_df = pd.read_csv(source_csv_path)
    approved_status_set = {'已审核通过', '已审核通过(历史迁移)'}
    if os.path.basename(source_csv_path) in ('finetune_candidates.csv', 'finetune_candidates_expanded.csv'):
        print(f"✅ 已加载审核通过候选集：{source_csv_path}")
        hard_examples = feedback_df
    elif '审核状态' in feedback_df.columns:
        hard_examples = feedback_df[(feedback_df['医生评价'].astype(str).str.contains('有误', na=False)) & (feedback_df['审核状态'].astype(str).isin(approved_status_set))]
    else:
        hard_examples = feedback_df[feedback_df['医生评价'].astype(str).str.contains('有误', na=False)]
    
    if len(hard_examples) == 0:
        print("✅ 目前医生反馈的诊断全部准确，模型无需微调！")
    else:
        print(f"📉 发现 {len(hard_examples)} 个医生反馈的困难样本，准备进行针对性微调...")
        
        # 疾病名称到数字的映射字典
        disease_classes = {'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3, 'akiec': 4, 'vasc': 5, 'df': 6}
                    
        class FeedbackDataset(Dataset):
            def __init__(self, dataframe):
                self.data = dataframe
                self.transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.RandomHorizontalFlip(), # 加点数据增强防止死记硬背
                    transforms.PILToTensor(),
                    transforms.ConvertImageDtype(torch.float32)
                ])
                
            def __len__(self):
                return len(self.data)
                
            def __getitem__(self, idx):
                row = self.data.iloc[idx]
                img_path = row['病例图片路径']
                # 提取实际正确的疾病简写 (比如从 "痣 (nv)" 中提取 "nv")
                label_str = row['最终修正标签'].split('(')[-1].replace(')', '')
                label_idx = disease_classes[label_str]
                
                image = Image.open(img_path).convert('RGB')
                image = self.transform(image)
                return image, label_idx
                
        class BaseSampleDataset(Dataset):
            def __init__(self, dataframe, image_dir):
                self.data = dataframe.reset_index(drop=True)
                self.image_dir = image_dir
                self.transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.RandomHorizontalFlip(),
                    transforms.PILToTensor(),
                    transforms.ConvertImageDtype(torch.float32)
                ])

            def __len__(self):
                return len(self.data)

            def __getitem__(self, idx):
                row = self.data.iloc[idx]
                img_path = os.path.join(img_dir, row['image_id'] + '.jpg')
                label_idx = int(row['label_idx'])

                image = Image.open(img_path).convert('RGB')
                image = self.transform(image)
                return image, label_idx

        feedback_dataset = FeedbackDataset(hard_examples)

        # 反馈样本很少时，必须混合原始训练样本，降低灾难性遗忘风险。
        base_to_hard_ratio = 6
        min_base_samples = 64
        max_base_samples = 1200
        available_base_df = train_df if 'train_df' in globals() else df
        base_sample_count = min(
            len(available_base_df),
            max(min_base_samples, min(max_base_samples, len(hard_examples) * base_to_hard_ratio))
        )
        base_sample_df = available_base_df.sample(n=base_sample_count, random_state=global_seed)
        base_dataset = BaseSampleDataset(base_sample_df, img_dir)
        mixed_dataset = ConcatDataset([feedback_dataset, base_dataset])
        mixed_loader = DataLoader(mixed_dataset, batch_size=8, shuffle=True)
        print(f"🧩 本次微调混合训练集：错题 {len(feedback_dataset)} 条 + 原始训练样本 {len(base_dataset)} 条 = {len(mixed_dataset)} 条")
        
        # 3. 加载模型
        model = ECAResNet50(num_classes=7).to(device) # ECAResNet50 
        best_model_path = model_v8_best_path
        model.load_state_dict(torch.load(best_model_path))
        
        def evaluate_val_metrics(eval_model, eval_loader, eval_device):
            eval_model.eval()
            eval_correct = 0
            eval_total = 0
            eval_logits_buffer = []
            eval_labels_buffer = []
            with torch.no_grad():
                for eval_images, eval_labels in eval_loader:
                    eval_images, eval_labels = eval_images.to(eval_device), eval_labels.to(eval_device)
                    eval_outputs = eval_model(eval_images)
                    _, eval_predicted = torch.max(eval_outputs.data, 1)
                    eval_total += eval_labels.size(0)
                    eval_correct += (eval_predicted == eval_labels).sum().item()
                    eval_logits_buffer.append(eval_outputs.detach().cpu())
                    eval_labels_buffer.append(eval_labels.detach().cpu())
            eval_acc = 100.0 * eval_correct / eval_total
            eval_logits_tensor = torch.cat(eval_logits_buffer, dim=0)
            eval_labels_tensor = torch.cat(eval_labels_buffer, dim=0)
            eval_precision, eval_recall, eval_f1, _, _, _ = evaluate_multiclass_metrics(eval_logits_tensor, eval_labels_tensor, list(disease_classes.keys()))
            return eval_acc, eval_precision * 100, eval_recall * 100, eval_f1 * 100

        baseline_val_acc, baseline_val_precision, baseline_val_recall, baseline_val_f1 = evaluate_val_metrics(model, val_loader, device)
        print(f"📌 微调前回归基线：Accuracy={baseline_val_acc:.2f}% | Recall={baseline_val_recall:.2f}% | F1={baseline_val_f1:.2f}%")
        min_save_acc = 87.50
        recall_baseline = 81.96
        f1_baseline = 79.91
        print(f"📏 反馈模型保存门槛：Accuracy >= {min_save_acc:.2f}%，且 Recall > {recall_baseline:.2f}% 或 F1 > {f1_baseline:.2f}%")

        # 4. 开启保守增量微调：小学习率 + 混合原始样本，防止破坏原有记忆
        criterion = nn.CrossEntropyLoss()
        # 学习率从平时的 0.0001 降到 0.000005
        optimizer = torch.optim.Adam(model.parameters(), lr=0.000005) 
        
        model.train()
        fine_tune_epochs = 2 # 反馈样本较少时保持少轮训练，降低过拟合和遗忘风险
        
        for epoch in range(fine_tune_epochs):
            total_loss = 0
            for images, labels in mixed_loader:
                images, labels = images.to(device), labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                
            print(f"  [微调训练] 第 {epoch+1}/{fine_tune_epochs} 轮混合学习... Loss: {total_loss:.4f}")
            
        post_val_acc, post_val_precision, post_val_recall, post_val_f1 = evaluate_val_metrics(model, val_loader, device)
        acc_delta = post_val_acc - baseline_val_acc
        print(f"📌 微调后回归验证：Accuracy={post_val_acc:.2f}% | Recall={post_val_recall:.2f}% | F1={post_val_f1:.2f}% | Accuracy变化={acc_delta:+.2f}%")
        if acc_delta < -1.0:
            print("⚠️ 回归测试未通过：微调后验证准确率下降超过 1%，存在灾难性遗忘风险。")
            print("🛑 为保护当前最优模型，本次反馈微调结果不会保存。请继续积累反馈样本或降低学习率后再试。")
        elif post_val_acc < min_save_acc:
            print(f"🛑 反馈模型 Accuracy {post_val_acc:.2f}% 低于最低保存门槛 {min_save_acc:.2f}%，本次结果不会保存。")
        elif not (post_val_recall > recall_baseline or post_val_f1 > f1_baseline):
            print(f"🛑 反馈模型 Recall={post_val_recall:.2f}%、F1={post_val_f1:.2f}% 未超过 v8 基准，本次结果不会保存。")
        else:
            print("✅ 保存条件通过：Accuracy 保持稳定，且 Recall 或 F1 超过 v8 基准。")

            # 5. 保存升级后的反馈微调模型
            new_model_path = model_v2_finetuned_path
            os.makedirs(os.path.dirname(new_model_path), exist_ok=True)
            torch.save(model.state_dict(), new_model_path)
            print(f"🎉 微调完成！模型已吸收错题经验。反馈增强模型已保存至：{new_model_path}")

🔄 正在启动【医疗 AI 错题本增量学习 (闭环演进)】系统...
✅ 已加载审核通过候选集：e:\AI\med_ai4\feedback_data1\finetune_candidates_expanded.csv
📉 发现 157 个医生反馈的困难样本，准备进行针对性微调...
🧩 本次微调混合训练集：错题 157 条 + 原始训练样本 942 条 = 1099 条
📌 微调前回归基线：Accuracy=87.97% | Recall=81.96% | F1=79.91%
📏 反馈模型保存门槛：Accuracy >= 87.50%，且 Recall > 81.96% 或 F1 > 79.91%
  [微调训练] 第 1/2 轮混合学习... Loss: 68.1097
  [微调训练] 第 2/2 轮混合学习... Loss: 54.8124
📌 微调后回归验证：Accuracy=86.97% | Recall=76.95% | F1=78.43% | Accuracy变化=-1.00%
🛑 反馈模型 Accuracy 86.97% 低于最低保存门槛 87.50%，本次结果不会保存。
